# 11 · Triangular Faults

*Carry the same linear inverse problem onto an unstructured, curved surface.*

**Learning objectives**

- explain when triangular elements are preferable to rectangles
- construct a small triangular fault in an explicit local frame
- reuse Green's assembly, inversion, regularization, and assessment unchanged
- choose azimuth or plate bases when patch strike varies

**Prerequisites:** Chapters 02 and 07  
**Estimated time:** 30–60 minutes

Notation follows the [glossary](../docs/glossary.md) and physical conventions
follow [conventions](../docs/conventions.md).


## Motivation

Rectangles tile a plane naturally but represent bends, slab surfaces, and
trace-following geometry awkwardly. Triangles tile arbitrary surfaces without
gaps. Their cost is unstructured patch orientation and neighborhood geometry;
their benefit is that the rest of the inverse problem remains $d=Gm$.


## Theory: same operator, different source basis

The Nikkhoo & Walter (2015) element is an artefact-free triangular dislocation
in an elastic half-space. Each triangle carries constant in-plane slip, exactly
as each rectangle did. Consequently only the kernel that fills columns of $G$
changes:

$$d=G_{tri}m+\epsilon.$$

Projection, covariance, weighted least squares, bounds, and posterior covariance
are unchanged. On a structured rectangle, the Laplacian is a grid stencil; on
an unstructured mesh it is built from geometric neighbors. Because each
triangle owns local strike and dip axes, one geographic `slip_azimuth` is often
more interpretable than one local rake.


## Build a tiny curved mesh

The nodes below are local East, North, Up meters tied to one `LocalFrame`.
Shared connectivity makes eight triangles. Real trace, polygon, and slab2.0
meshing stays in `examples/mesh_generation.ipynb`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

rng = np.random.default_rng(0)
frame = geodef.LocalFrame(34.0, -118.0)
nodes = np.array([
    [-12_000.0, -6_000.0, -3_000.0],
    [0.0, -6_000.0, -4_000.0],
    [12_000.0, -6_000.0, -5_500.0],
    [-12_000.0, 0.0, -5_500.0],
    [0.0, 0.0, -6_800.0],
    [12_000.0, 0.0, -8_500.0],
    [-12_000.0, 6_000.0, -8_000.0],
    [0.0, 6_000.0, -9_500.0],
    [12_000.0, 6_000.0, -12_000.0],
])
triangles = np.array([
    [0, 1, 3], [1, 4, 3], [1, 2, 4], [2, 5, 4],
    [3, 4, 6], [4, 7, 6], [4, 5, 7], [5, 8, 7],
])
fault = geodef.Fault.from_triangles(nodes, frame=frame, triangles=triangles)
print(fault.validate())
geodef.plot.fault3d(fault, color_by="depth", aspect=2.0);


The mesh deepens northward and bends slightly eastward. Patch strike
and dip vary, so a common local rake would not represent exactly one geographic
direction. We prescribe eastward azimuth and let GeoDef convert it patch by
patch.


In [ ]:
amplitude_true = np.array([0.15, 0.35, 0.65, 0.95, 0.3, 0.7, 0.9, 0.4])
strike_true, dip_true = geodef.slip.from_azimuth(
    amplitude_true,
    azimuth_degrees=90.0,
    fault_strike_degrees=fault.strike,
)
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.18, -117.82, 6), np.linspace(33.90, 34.16, 5)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=strike_true, slip_dip=dip_true
)
sigma = 0.003
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    name="triangular_gnss",
)
result = geodef.solve(
    fault, datasets=gnss, components="azimuth", slip_azimuth=90.0,
    regularization="laplacian", regularization_strength=0.1,
    bounds=(0.0, None),
)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
geodef.plot.slip(fault, amplitude_true, ax=axes[0], title="Input amplitude", colorbar_label="Slip (m)")
geodef.plot.slip_interpolated(
    fault, result.slip_magnitude, ax=axes[1], title="Recovered, interpolated", colorbar_label="Slip (m)"
);


The solver call is the same one used for rectangles. Interpolation
makes a readable surface but creates no new resolution; the eight recovered
patch values remain the actual estimate.


## Checkpoint questions

**Which parts of the inverse problem changed?**
<details><summary>Answer</summary>The source geometry and dislocation kernel;
the linear algebra, covariance, constraints, and assessment did not.</details>

**Why prefer azimuth over one rake here?**
<details><summary>Answer</summary>Patch strike varies, so a common local rake
would rotate in geographic space.</details>

**Does a smooth interpolated plot imply resolved smooth slip?**
<details><summary>Answer</summary>No. It is a rendering of discrete patch values.</details>


## Common mistakes

- Supplying depth-positive-down as local `z` places triangles above the surface;
  ENU `up` is negative at depth.
- Combining vertices built in different local frames silently destroys geometry.
- Smoothing triangle-local strike/dip components across a sharp bend can impose
  an artificial prior direction.


## Recap

- Triangles represent curved surfaces while retaining piecewise-constant slip.
- The public forward, solve, and assessment workflow is geometry-independent.
- Direction bases must be interpreted in geographic as well as local axes.

Use the geometry and slip-basis guides in [workflow](../docs/workflow.md).


## Exercises

Worked solutions are in `solutions/11_triangular_faults_solutions.ipynb`.

1. Split each triangle and compare prediction convergence and conditioning.
2. Compare fixed rake with fixed azimuth and map their geographic directions.
3. Run a spike test on each triangle and compare localization with depth.
4. Challenge: add a second segment with a sharp bend and discuss whether the
   Laplacian should connect patches across the join.


## Further reading

- Nikkhoo & Walter (2015), triangular dislocations in an elastic half-space.
- Meade (2007), triangular dislocation elements for complex faults.
- `examples/mesh_generation.ipynb`, practical trace/polygon/slab meshing.
